# Preprocessing & Feature Engineering

Compute feature columns such as log return, rolling volatility, moving averages, and a simple trend label.

In [5]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure the notebook runs from the project root
project_root = Path(r"D:/BTLDATA/DATA_MINING_PROJECT")
os.chdir(project_root)

from src.data import loader

# Load all raw crypto CSVs
(df) = loader.load_all_crypto_data("data/raw")

# Normalize datetime column name
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.rename(columns={"date": "Date"})

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Required columns
assert "Close" in df.columns, "Expected Close column"

# Sort data for rolling computations
if "coin" in df.columns:
    df = df.sort_values(["coin", "Date"]).reset_index(drop=True)
else:
    df = df.sort_values("Date").reset_index(drop=True)

# Feature engineering
# 1) log return
# Using log(C_t / C_{t-1}) is numerically stable and equivalent to log(C_t) - log(C_{t-1})
df["log_return"] = df.groupby("coin")["Close"].transform(lambda x: np.log(x / x.shift(1)))

# 2) rolling volatility (7-day, based on log return)
df["vol_7d"] = df.groupby("coin")["log_return"].transform(lambda x: x.rolling(7, min_periods=1).std())

# 3) moving averages (7-day and 30-day)
for window in [7, 30]:
    df[f"ma_{window}d"] = df.groupby("coin")["Close"].transform(lambda x: x.rolling(window, min_periods=1).mean())

# 4) trend label based on daily log return
# Use np.where for speed and clarity
df["trend_up_down"] = np.where(
    df["log_return"] > 0,
    "up",
    np.where(df["log_return"] < 0, "down", "flat"),
)

# 5) volatility regime based on vol_7d median per coin
median_vol = df.groupby("coin")["vol_7d"].transform("median")
df["vol_regime"] = np.where(df["vol_7d"] > median_vol, "high_vol", "low_vol")

# Drop rows with NaN values
df.dropna(inplace=True)

# Quick sanity check
print("Number of unique coins:", df["coin"].nunique())

# Save processed features to parquet for downstream use
processed_path = Path("data/processed/crypto_features.parquet")
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path, index=False)

# Display example rows
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()


Number of unique coins: 23
Shape: (37036, 17)
Columns: ['SNo', 'Name', 'Symbol', 'Date', 'High', 'Low', 'Open', 'Close', 'Volume', 'Marketcap', 'coin', 'log_return', 'vol_7d', 'ma_7d', 'ma_30d', 'trend_up_down', 'vol_regime']


,SNo,Name,Symbol,Date,High,Low,Open,Close,Volume,Marketcap,coin,log_return,vol_7d,ma_7d,ma_30d,trend_up_down,vol_regime
2,3,Aave,AAVE,2020-10-07 23:59:59,42.408314,35.970690,42.399947,40.083976,6.828342e+05,6.713004e+07,coin_Aave,-0.056209,0.120933,45.234939,45.234939,down,high_vol
3,4,Aave,AAVE,2020-10-08 23:59:59,44.902511,36.696057,39.885262,43.764463,1.658817e+06,2.202651e+08,coin_Aave,0.087845,0.157732,44.867320,44.867320,up,high_vol
4,5,Aave,AAVE,2020-10-09 23:59:59,47.569533,43.291776,43.764463,46.817744,8.155377e+05,2.356322e+08,coin_Aave,0.067440,0.144860,45.257405,45.257405,up,high_vol
5,6,Aave,AAVE,2020-10-10 23:59:59,51.405655,46.703328,46.818146,49.133718,1.074627e+06,2.472884e+08,coin_Aave,0.048283,0.130494,45.903457,45.903457,up,high_vol
6,7,Aave,AAVE,2020-10-11 23:59:59,51.453374,48.716040,49.133130,49.660726,6.921506e+05,2.499408e+08,coin_Aave,0.010669,0.117223,46.440210,46.440210,up,high_vol
